In [ ]:
import sys
from pathlib import Path

PYTHON_DIRECTORY = Path("python").resolve()
if str(PYTHON_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIRECTORY))

from hecke_congruences import *
from load_source_data import load_source_data

USE_ARCHIVED_SOURCE_DATA = True
SOURCE_DATA_DIRECTORY = Path("source_data")
MOD27_SOURCE_ARCHIVE = (
    SOURCE_DATA_DIRECTORY / "p3_mod27_T2_T7_all_degrees.npz"
)
MOD27_ZERO_SOURCE_ARCHIVE = (
    SOURCE_DATA_DIRECTORY / "p3_mod27_zero_T2_mod243.npz"
)

def get_source_data(R, d, q, archive):
    if USE_ARCHIVED_SOURCE_DATA:
        return load_source_data(R, d, q, archive)

    return prepare_source_data(R, d, q)

In [10]:
p = 3
m = 3
modulus = p^m
R = Integers(modulus)

S.<X> = PolynomialRing(R)

period = euler_phi(p^m)
a_m = p^m * (p - 1)
b_m = p^(m - 1)*(p + 1)
surjectivity_bound = a_m + b_m


base_degrees = tuple(d for d in range(0, a_m + b_m, 2) if d%6 == 2)

degree_residues = tuple(d%period for d in base_degrees)

F2 = {
    r: X^3 - X
    for r in degree_residues
}

Q2 = X

exact_induction_base = tuple(
    d
    for d in range(
        p**(m - 1)*(p + 1),
        a_m + b_m,
        2,
    )
    if d%6 == 2
)

print("Dickson degrees:", a_m, b_m)
print("full finite verification range:", base_degrees)
print("exact induction base:", exact_induction_base)

Dickson degrees: 54 36
full finite verification range: (2, 8, 14, 20, 26, 32, 38, 44, 50, 56, 62, 68, 74, 80, 86)
exact induction base: (38, 44, 50, 56, 62, 68, 74, 80, 86)


In [12]:
def verify_zero_branch_case(d, q):
    """
    Verify the zero-branch staged T2 identity in one degree and
    orientation.

    Source data are loaded from the archive or computed afresh,
    according to USE_ARCHIVED_SOURCE_DATA.
    """
    data = get_source_data(
        R,
        d,
        q,
        MOD27_SOURCE_ARCHIVE,
    )

    return verify_divided_identities(
        F=F2[d%period],
        Q=Q2,
        n=2,
        a=2,  # divide T_2 by 3^2 = 9
        b=1,  # test Z^3-Z modulo 3
        data=data,
        check_descent=False,
    )


lower_base_degrees = tuple(
    d
    for d in base_degrees
    if d not in exact_induction_base
)

ordered_degrees = (
    tuple(sorted(exact_induction_base, reverse=True))
    + tuple(sorted(lower_base_degrees, reverse=True))
)

cases = [
    (d, q)
    for d in ordered_degrees
    for q in range(0, p - 1)
]
results = []

for d, q in cases:
    test = verify_zero_branch_case(d, q)
    results.append(test)

    print(
        f"d={d:3d}, "
        f"r={test['residue']:3d}, "
        f"q={q}, "
        f"sign={test['sign']:+d}, "
        f"rank={test['rank']:3d}, "
        f"division={test['division_passed']}, "
        f"terminal={test['terminal_passed']}, "
        f"passed={test['passed']}"
    )

    assert test["passed"]

print("——————————————————————————————————————————————")
print("cases completed:", len(results))
print("ALL ZERO-BRANCH IDENTITIES VERIFIED")

d= 86, r= 14, q=0, sign=+1, rank= 11, division=False, terminal=False, passed=False


AssertionError: 

In [24]:
M = ModularSymbols(1, 88, sign=1)
T2 = M.integral_hecke_matrix(2)

In [41]:
C = (T2^3 - 81*T2)/729
C%3

[0 0 2 0 2 2 0 0]
[0 0 2 2 0 1 2 0]
[0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0]

In [37]:
3^6

729